In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, round, current_timestamp


SOURCE_MAPPING = [
    ("customers", "bronze_customers", "silver_customers", "rejected_customers"),
    ("addresses", "bronze_addresses", "silver_addresses", "rejected_addresses"),
    ("orders", "bronze_orders", "silver_orders", "rejected_orders"),
    ("payments", "bronze_payments", "silver_payments", "rejected_payments"),
    ("support_tickets", "bronze_tickets", "silver_tickets", "rejected_tickets"),
    ("loyalty", "bronze_loyalty", "silver_loyalty", "rejected_loyalty"),
]


def table_count(table_name):
    return spark.table(table_name).count()


rows = []

for source_name, bronze_table, silver_table, rejected_table in SOURCE_MAPPING:
    bronze_rows = table_count(bronze_table)
    silver_rows = table_count(silver_table)
    rejected_rows = table_count(rejected_table)

    total_processed = silver_rows + rejected_rows

    rows.append(
        Row(
            source_name=source_name,
            bronze_rows=bronze_rows,
            silver_valid_rows=silver_rows,
            silver_rejected_rows=rejected_rows,
            total_processed_rows=total_processed
        )
    )


dq_df = spark.createDataFrame(rows)

dq_df = (
    dq_df
    .withColumn(
        "rejection_rate_percent",
        round(
            (col("silver_rejected_rows") / col("total_processed_rows")) * 100,
            2
        )
    )
    .withColumn("_gold_processed_at", current_timestamp())
)

(
    dq_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_data_quality_summary")
)

display(dq_df)

source_name,bronze_rows,silver_valid_rows,silver_rejected_rows,total_processed_rows,rejection_rate_percent,_gold_processed_at
customers,102000,99048,952,100000,0.95,2026-07-06T12:23:49.607Z
addresses,101000,97546,2454,100000,2.45,2026-07-06T12:23:49.607Z
orders,484628,469825,10005,479830,2.09,2026-07-06T12:23:49.607Z
payments,446598,430303,12751,443054,2.88,2026-07-06T12:23:49.607Z
support_tickets,75857,73623,1483,75106,1.97,2026-07-06T12:23:49.607Z
loyalty,55550,53720,1280,55000,2.33,2026-07-06T12:23:49.607Z


In [0]:
%sql
RESTORE TABLE gold_customer_360 TO VERSION AS OF 2;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
3052567,1,1,1,3052505,3052567
